# wandb-log-step composite — cx15: log scalar loss to wandb with step kwarg

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `backward-on-scalar-loss`, `wandb-log-step`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "wandb-log-step"
DD_ATOM_IDS = ["backward-on-scalar-loss", "wandb-log-step"]
DD_SUBTOPICS = ["PyTorch: backward()", "Logging: wandb.log step"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The number you log to wandb is the SAME number you call `.backward()` on — modulo `.item()`. The composition makes the order explicit:

```python
loss = loss_fn(model(x), y)     # scalar.
loss.backward()                  # atom A: backward-on-scalar-loss.
wandb.log({'loss': loss.item()}, step=step_count)  # atom B: wandb-log-step.
```

**Atom A — `backward-on-scalar-loss`.** Backward requires a scalar; same atom as cx13/cx14.

**Atom B — `wandb-log-step`.** `wandb.log(metrics_dict, step=int)` sends one row to the dashboard. The `step` kwarg is the x-axis. Pass `loss.item()` (Python float), NOT the tensor — wandb serializes to JSON and tensors aren't JSON-able. If you omit `step`, wandb uses its own monotonic counter, which makes runs with different batch sizes uncomparable.

**Why both atoms in the same drill.** A common bug: log `loss` instead of `loss.item()`. It still 'works' in the sense that the JSON serializer raises later, far from where the logic broke. We test by mocking `wandb` and inspecting `wandb.log.call_args_list` — every logged value MUST be a plain Python `float`, every step MUST be a plain Python `int`.

### Composite Exercise — log scalar loss to wandb with step kwarg

**Atoms exercised together**: `backward-on-scalar-loss`, `wandb-log-step`

Implement `cx15_train_and_log(model, optimizer, loader, loss_fn, batch_size)`. ONE epoch.

For each `(x, y)` in `loader`, batch index `i` starting at 0:
1. `pred = model(x)`; `loss = loss_fn(pred, y)`.
2. `loss.backward()` (atom A).
3. `optimizer.step()`; `optimizer.zero_grad()`.
4. Compute `examples_seen = (i + 1) * batch_size`.
5. Call `wandb.log({'loss': loss.item()}, step=examples_seen)` (atom B).

Return the final `examples_seen`.

**Test asserts (via mocked wandb)**:
- One `wandb.log` call per batch.
- Each call's positional first arg is a `dict` with key `'loss'` mapping to a Python `float` (NOT a `Tensor`).
- Each call's `step` kwarg is an `int` equal to `(i+1)*batch_size`.
- Final returned value equals `len(loader)*batch_size`.

In [ ]:
def cx15_train_and_log(model, optimizer, loader, loss_fn, batch_size):
    examples_seen = 0
    for i, (x, y) in enumerate(loader):
        pred = model(x)
        loss = loss_fn(pred, y)
        # Atom A: scalar-loss backward.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        examples_seen = (i + 1) * batch_size
        # Atom B: wandb.log with step kwarg. loss.item() — NOT the tensor.
        wandb.log({'loss': loss.item()}, step=examples_seen)
    return examples_seen


<details><summary>Show solution — cx15</summary>

```python
def cx15_train_and_log(model, optimizer, loader, loss_fn, batch_size):
    examples_seen = 0
    for i, (x, y) in enumerate(loader):
        pred = model(x)
        loss = loss_fn(pred, y)
        # Atom A: scalar-loss backward.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        examples_seen = (i + 1) * batch_size
        # Atom B: wandb.log with step kwarg. loss.item() — NOT the tensor.
        wandb.log({'loss': loss.item()}, step=examples_seen)
    return examples_seen
```

Computing `examples_seen` from `(i+1)*batch_size` instead of `len(x)*(i+1)` matters for the last partial batch — when `drop_last=False` the final batch may be smaller. ARENA's convention is the simple `(i+1)*batch_size` form because it makes runs with the same loader configuration directly comparable; production code often uses the exact `examples_seen += x.size(0)` form instead.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx15'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx15',
        'subtopics': ["PyTorch: backward()", "Logging: wandb.log step"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()